# 📊 02｜便利商店一週銷售：先算營收，再比較類別

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/PythonForDataAnalysis/blob/master/AI%E5%8D%94%E4%BD%9C%E8%B3%87%E6%96%99%E5%88%86%E6%9E%90_12%E5%80%8B%E7%AF%84%E4%BE%8B/02_store_sales/lesson.ipynb)

> 💡 **學習模式說明**：
> 本筆記本專為**完全不寫程式的初學者**設計，全程搭配 **Google Colab 內建的 Gemini AI** 進行協作。
> 你不需要手寫程式碼：先用 Prompt 指揮 AI，再對照下方已附的參考程式，執行並驗收數字與圖表。
> 🆕 **本例只新增一個重點**：原始資料沒有「營收」欄，要先用 **單價 × 數量** 衍生出新指標，才能回答營收問題。

## 🪜 本例路線：先完成一次，再逐步加深
1. **必做主線（單元一～三）**：看懂 175 列銷售紀錄、逐列算出營收，最後用**一張垂直長條圖**回答「哪一類的整週營收最高？」。單元三後先停下來驗收。
2. **報表應用（單元四）**：把類別營收與品項明細匯出成 Excel，重新讀檔核對總營收。
3. **選做挑戰（單元五～六）**：看每日營收的走勢，或找出「異常的一天」。第一次上課不必全部完成。

---

## 🧭 資料專案介紹：貫徹「看懂資料與選對圖表」思維 SOP

資料：`data/store_sales.csv`（請先閱讀[本例資料卡](./data/DATA_CARD.md)）。**一列代表：某一天、某一個品項的銷售紀錄**，共 7 天 × 25 個品項＝175 列。不得為了符合預想結果而改動原檔。

### 1. 欄位角色貼標籤
- 🔵 **條件 (Dimension)**：`日期`（時間）、`品項`、`類別`（類別）➔ 代表「哪一天、哪一類」，用來分組。
- 🟢 **指標 (Metric)**：`單價`（元）、`數量`（件）➔ 兩個都是數值，但**單價不能直接加總**。
- 🟢 **衍生指標**：`營收`＝單價 × 數量 ➔ 原檔沒有，必須先算出來才能回答營收問題。
- ⚪ **標識 (ID)**：本例沒有流水號；`品項` 是商品名稱，只用來分組，不做數值運算。

### 2. 五大提問積木與核心問題
- 🎯 **比較（必做）**：哪一類的整週營收最高？哪一類最低？
- 📈 **趨勢（選做）**：這 7 天的每日總營收如何變化？
- 🔍 **比較＋邊界（選做）**：有沒有哪一天某個品項賣得特別多？這份資料能不能說明原因？

### 3. 邊界反向檢核防呆（不可觸犯的紅線）
- ⚠️ **缺指標先衍生**：原檔沒有「營收」欄；只看數量會回答成「哪類賣最多件」，不是「哪類賺最多」。
- ⚠️ **單價不能加總**：把所有單價加起來不代表任何意義；營收要逐列「單價 × 數量」再加總。
- ⚠️ **期間太短**：只有一週 7 天，不能推論長期銷售趨勢或季節性。
- ⚠️ **缺欄位不推原因**：資料中**沒有天氣欄位**，看到某天雨傘大賣，也不能只靠這張表斷定是下雨造成。

### 4. 決定最適圖表
- **5 個類別比營收高低** ➔ **垂直長條圖**（類別少、名稱短；Y 軸從 0 開始，單位「元」）。
- **每日總營收走勢（選做）** ➔ **折線圖**（X 軸日期由早到晚，只有 7 個點）。
- **雨傘每日銷量（選做）** ➔ **垂直長條圖**，把異常的那一天用醒目顏色標出。

---

## 🤖 Colab Gemini AI 操作方式
1. 每次只複製一個單元的 Prompt，貼到 Colab Gemini 對話框。
2. 比較 AI 的答覆與下方參考程式，執行目前單元的儲存格。
3. 先看懂輸出並回答驗收問題，再進入下一單元。

In [ ]:
# 第 0 步：下載範例資料並安裝繁體中文字型（請先點擊左側 ▶ 執行此儲存格）
import os
from pathlib import Path
from urllib.request import urlretrieve

# 1. 下載資料（已存在就不重複下載；若下載失敗，請手動上傳到 data/）
data_dir = Path('data')
data_dir.mkdir(exist_ok=True)
files = {
    'store_sales.csv': 'https://raw.githubusercontent.com/roberthsu2003/PythonForDataAnalysis/master/AI%E5%8D%94%E4%BD%9C%E8%B3%87%E6%96%99%E5%88%86%E6%9E%90_12%E5%80%8B%E7%AF%84%E4%BE%8B/02_store_sales/data/store_sales.csv',
}
for name, url in files.items():
    if not (data_dir / name).exists():
        urlretrieve(url, data_dir / name)
print('📂 data/ 內的檔案：', sorted(p.name for p in data_dir.iterdir()))

# 2. 安裝與設定 Colab 繁體中文字型 (思源黑體)
%pip install -q mplfonts
import glob
import mplfonts
from matplotlib import font_manager
from mplfonts import use_font

for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
    font_manager.fontManager.addfont(f)
use_font("Noto Sans CJK SC")

print("✅ 環境初始化完成！資料已就緒，繁體中文字型已設定成功。請開始進行下方單元。")

### 📝 單元一【必做】：讀取資料與欄位角色檢核（貼標籤）

> 🎯 **思維重點**：先確認「一列代表什麼」，再分出條件與指標；並發現資料裡**沒有營收欄**。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請幫我用 pandas 讀取 data/store_sales.csv：
1. 顯示前 5 筆資料、所有欄位名稱與總列數。
2. 檢查各欄位的資料型態，並統計每一欄有多少缺值。
3. 計算資料有幾個不同日期、幾個不同品項，說明「一列代表什麼」。
4. 用文字回報：哪些欄位是條件（時間／類別）、哪些是數值指標；資料中有沒有「營收」欄位？
先不要畫圖，也不要下結論。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
import pandas as pd

df = pd.read_csv('data/store_sales.csv')
print("--- 前 5 筆資料 ---")
display(df.head())

print(f"\n總列數：{len(df)}；不同日期 {df['日期'].nunique()} 天 × 不同品項 {df['品項'].nunique()} 個")
print("➡️ 一列代表：某一天、某一個品項的銷售紀錄")

print("\n--- 欄位型態 ---")
df.info()
print("\n各欄缺值數：")
print(df.isna().sum())
print("\n資料中有『營收』欄位嗎？", '營收' in df.columns)

### 📝 單元二【必做】：邊界檢核：衍生「營收」指標，對照數量排名

> 🎯 **思維重點**：問題問「營收」，但原檔沒有這欄：逐列算出「單價 × 數量」，並對照「數量排名」與「營收排名」是否相同。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
核心問題是「哪一類的整週營收最高？」，但原檔沒有營收欄位：
1. 新增「營收」欄位，逐列計算「單價 × 數量」；請在註解說明為什麼不能把單價直接加總。
2. 依「類別」分組，計算整週銷售數量合計與整週營收合計。
3. 在同一張表加上「數量排名」與「營收排名」，依營收由高到低排序。
4. 核對：逐列營收的總和，是否等於各類別營收加總？
不要修改原始 CSV 檔。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
# 原檔沒有營收欄：逐列「單價 × 數量」才是營收。
# 單價直接加總沒有意義（等於把每件售價疊在一起，不代表賣了多少錢）。
df['營收'] = df['單價'] * df['數量']

category_df = df.groupby('類別').agg(銷售數量=('數量', 'sum'), 整週營收=('營收', 'sum'))
category_df['數量排名'] = category_df['銷售數量'].rank(ascending=False).astype(int)
category_df['營收排名'] = category_df['整週營收'].rank(ascending=False).astype(int)
category_df = category_df.sort_values('整週營收', ascending=False)
display(category_df)

print(f"逐列營收合計：{df['營收'].sum():,} 元")
print(f"各類別營收加總：{category_df['整週營收'].sum():,} 元")
print("兩者一致：", df['營收'].sum() == category_df['整週營收'].sum())

### 📝 單元三【必做】：提問積木【比較】— 哪一類的整週營收最高？（垂直長條圖）

> 🎯 **思維重點**：問題類別屬於「比較」，5 個類別比高低，依選圖規則挑選【垂直長條圖】，Y 軸從 0 開始並標示單位「元」。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請用 matplotlib 畫出 5 個類別「整週營收」的垂直長條圖：
1. X 軸為類別，Y 軸為整週營收（元），柱子依營收由高到低排列。
2. Y 軸數值起點必須從 0 開始。
3. 在柱子上方標註營收金額（加千分位逗號）。
4. 營收最高的類別用醒目顏色標示，並寫在標題中。
5. 圖表標題、座標軸與標籤全部使用繁體中文；圖上數字要和上一步的表格一致。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
import matplotlib.pyplot as plt

revenue = category_df['整週營收']
top_category = revenue.idxmax()
colors = ['#e74c3c' if c == top_category else '#3498db' for c in revenue.index]

plt.figure(figsize=(8, 5))
bars = plt.bar(revenue.index, revenue.values, color=colors, width=0.55)
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, yval + 1000, f'{yval:,.0f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.ylim(0, revenue.max() * 1.15)
plt.title(f'各類別整週營收比較（最高：{top_category}）', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('商品類別', fontsize=12)
plt.ylabel('整週營收（元）', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

## ✅ 必做主線驗收：現在先停下來

✏️ **先不要看答案！** 對照上面的表格和圖，自己回答下列問題，把你的答案寫在紙上或筆記裡：

- [ ] 能說出一列代表「一天 × 一個品項」，並指出單價、數量是指標；營收是自己衍生的指標。
- [ ] 能從表格與長條圖指出整週營收最高與最低的類別，並核對兩處數字一致；圖的數值軸從 0 開始。
- [ ] 能用一句話解釋：為什麼「賣最多件」的類別排名，和「營收最高」的類別排名可能不同？
- 💬 **討論題**：零食賣出的件數比日用品多，為什麼營收反而比較少？

<details>
<summary>👉 寫好了嗎？點此展開參考答案核對</summary>

- 資料共 175 列、沒有缺值；營收逐列合計＝分類加總＝ **236,096 元**。
- 長條圖與表格都顯示**鮮食**整週營收最高（77,491 元）、**冰品**最低（3,465 元）；數值軸從 0 開始。

</details>

若今天繼續做報表，請進行**單元四**；單元五、六是選做挑戰。

---

### 📝 單元四【報表應用】：匯出類別營收 Excel 報表

> 🎯 **思維重點**：把已驗證的結果交付出去；匯出後**重新讀檔**，確認總營收沒有在匯出時改變。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請將分析結果匯出成 Excel 檔案：
1. 存為 output/類別營收報表.xlsx（若 output 目錄不存在請自動建立）。
2. 第一個工作表「類別營收」：類別、銷售數量、整週營收、數量排名、營收排名。
3. 第二個工作表「品項明細」：原始欄位加上營收欄位。
4. 匯出後重新讀取兩個工作表，核對列數與營收合計是否與原始資料一致。
5. 提供 Colab 下載該 Excel 檔案的程式碼。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
os.makedirs('output', exist_ok=True)
output_path = 'output/類別營收報表.xlsx'
with pd.ExcelWriter(output_path) as writer:
    category_df.reset_index().to_excel(writer, sheet_name='類別營收', index=False)
    df.to_excel(writer, sheet_name='品項明細', index=False)

# 重新讀檔核對：匯出前後的數字必須一樣
check_category = pd.read_excel(output_path, sheet_name='類別營收')
check_detail = pd.read_excel(output_path, sheet_name='品項明細')
print('類別營收：', len(check_category), '列；營收合計', f"{check_category['整週營收'].sum():,}", '元')
print('品項明細：', len(check_detail), '列；營收合計', f"{check_detail['營收'].sum():,}", '元')

try:
    from google.colab import files
    files.download(output_path)
    print("✅ 已啟動瀏覽器下載 Excel 報表！")
except Exception:
    print(f"✅ Excel 檔案已儲存於：{output_path}")

### 📝 單元五【選做挑戰】：提問積木【趨勢】— 每日總營收如何變化？（折線圖）

> 🎯 **思維重點**：「時間 ＋ 指標」問演變走向，用【折線圖】，X 軸由早到晚。邊界提醒：只有 7 個點，只能描述這一週，不能主張長期趨勢。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請依日期彙總每日總營收，並畫出折線圖：
1. 將日期轉成日期格式並由早到晚排序，X 軸標示「月/日（星期）」。
2. 每個資料點加上圓點標記與營收數字（加千分位）。
3. Y 軸從 0 開始，標示單位「元」，標題與座標軸使用繁體中文。
4. 用文字指出哪一天營收最高，並提醒：只有 7 天資料，不足以主張長期趨勢。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
daily = df.groupby('日期', as_index=False)['營收'].sum()
daily['日期'] = pd.to_datetime(daily['日期'])
daily = daily.sort_values('日期')
weekday_names = ['一', '二', '三', '四', '五', '六', '日']
labels = [f"{d.month}/{d.day}（{weekday_names[d.weekday()]}）" for d in daily['日期']]

plt.figure(figsize=(9, 5))
plt.plot(labels, daily['營收'], marker='o', color='#3498db', linewidth=2)
for x, y in zip(labels, daily['營收']):
    plt.text(x, y + 800, f'{y:,}', ha='center', fontsize=10)
plt.ylim(0, daily['營收'].max() * 1.2)
plt.title('一週每日總營收走勢', fontsize=14, fontweight='bold')
plt.xlabel('日期')
plt.ylabel('每日總營收（元）')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

best = daily.loc[daily['營收'].idxmax()]
print(f"💡 營收最高的一天：{best['日期']:%m/%d}，共 {best['營收']:,} 元")
print("⚠️ 邊界提醒：只有 7 天資料，只能描述這一週，不能推論長期趨勢或季節性。")

### 📝 單元六【選做挑戰】：找出「異常的一天」，但不亂推原因（垂直長條圖）

> 🎯 **思維重點**：用篩選找出單日銷量明顯偏高的品項；資料中**沒有天氣欄位**，只能描述現象，不能斷定原因。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請找出這一週「單日銷量特別突出」的品項：
1. 對每個品項計算 7 天的平均銷量，找出「單日數量超過自己平均 2 倍」的紀錄。
2. 針對找到的品項，畫出 7 天每日銷量的垂直長條圖，Y 軸從 0 開始，突出的那一天用醒目顏色標示。
3. 用文字描述「哪一天、哪個品項、賣了多少」。
4. 檢查資料中是否有天氣欄位；若沒有，請明確說明：這份資料無法證明銷量突出的原因。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
item_mean = df.groupby('品項')['數量'].transform('mean')
spikes = df[df['數量'] > item_mean * 2]
display(spikes[['日期', '品項', '類別', '數量']])

item = spikes['品項'].iloc[0]
item_daily = df[df['品項'] == item].set_index('日期')['數量']
spike_day = item_daily.idxmax()
colors = ['#e74c3c' if d == spike_day else '#95a5a6' for d in item_daily.index]

plt.figure(figsize=(8, 4.5))
plt.bar([d[5:] for d in item_daily.index], item_daily.values, color=colors)
for x, y in zip([d[5:] for d in item_daily.index], item_daily.values):
    plt.text(x, y + 0.3, str(y), ha='center')
plt.ylim(0, item_daily.max() * 1.2)
plt.title(f'{item} 每日銷量（{spike_day} 特別突出）', fontsize=13, fontweight='bold')
plt.xlabel('日期（月-日）')
plt.ylabel('銷量（件）')
plt.show()

print("資料中有天氣欄位嗎？", '天氣' in df.columns)
print(f"⚠️ 邊界提醒：只能說 {spike_day} 的{item}銷量特別高；這份資料沒有天氣欄位，不能據此證明是下雨造成。")

---

## 🎯 學習驗收與成果回顧

完成目前選擇的層次後，請檢視你是否能親自驗收：
- [ ] **必做**：指出條件與指標、自行衍生營收，核對鮮食整週營收最高、長條圖與表格一致。
- [ ] **報表應用**：Excel 重新讀取後，營收合計仍為 236,096 元。
- [ ] **選做挑戰**：折線圖只描述這一週；異常的一天只描述現象，不推論原因。
- [ ] **AI 協作**：能說明自己提出的問題、AI 幫忙執行的事，以及自己如何確認結果。